In [1]:
import functools
import os
import sys
import json

from ultralytics import YOLO
import pandas as pd

sys.path.append("..")
import ultralytics_validator_wrapper
from tflite_inference import TFLiteInferer, TFLiteMerged
from tflite_inference_baseline import TFLiteMerged as TFLiteInfererCorrected

pd.options.display.float_format = '{:.3f}'.format

%matplotlib inline
%load_ext autoreload
%autoreload 2

TensorFlow version:  2.17.0
TensorFlow version:  2.17.0


In [2]:
BASE_EXPERIMENT_DIR = "../results/cam3_revB_batch1_7_margin_15_yolov8n_bg_wb_alif_clahe_box_01_dfl_01_2nd"
BASE_WEIGHTS_DIR = os.path.join(BASE_EXPERIMENT_DIR, "weights")
WEIGHTS_PTH = os.path.join(BASE_WEIGHTS_DIR, "best.pt")

HOLDOUT_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/project_holdout.yaml"

if os.path.exists(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json")):
    with open(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json"), "r") as f:
        data = json.load(f)
    CONF_TH = float(data["conf_th"])
    IOU_TH = float(data["iou_th"])
else:
    print("No inference_params.json found, using default thresholds.")
    CONF_TH = 0.35
    IOU_TH = 0.50

print(f"Using CONF_TH={CONF_TH}, IOU_TH={IOU_TH}")

args = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=IOU_TH,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

args_reduced_iou = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=0.35,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

Using CONF_TH=0.2, IOU_TH=0.5


In [3]:
model = YOLO(WEIGHTS_PTH)

In [4]:
inferer = TFLiteInferer(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
    head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

inferer_merged = TFLiteMerged(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

inferer_merged_iou = TFLiteMerged(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=0.35,
    imgsz=320,
)

inferer_merged_bias = TFLiteInfererCorrected(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
    use_bias=True,
)

inferer_merged_bias_iou = TFLiteInfererCorrected(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=0.35,
    imgsz=320,
    use_bias=True,
)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
args["data"] = HOLDOUT_PTH
args_reduced_iou["data"] = HOLDOUT_PTH

print("BASELINE")
ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

print("MERGED")
ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_merged, args),
    False,
)

print("MERGED REDUCED IOU")
ultralytics_validator_wrapper.validate_model(
    args_reduced_iou,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_merged_iou, args_reduced_iou),
    False,
)

BASELINE


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]
tst validation:   0%|          | 0/2 [00:00<?, ?it/s]

tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.38s/it]


                   all        148        409      0.932      0.878      0.928      0.732      0.951      0.895      0.943       0.94
   task    F1     P     R  MAP@50
0   box 0.904 0.932 0.878   0.928
1  pose 0.922 0.951 0.895   0.943
MERGED


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]
tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.24s/it]


                   all        148        409      0.613      0.772      0.637      0.204       0.78      0.853      0.853      0.848
   task    F1     P     R  MAP@50
0   box 0.683 0.613 0.772   0.637
1  pose 0.815 0.780 0.853   0.853
MERGED REDUCED IOU


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]
tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.21s/it]


                   all        148        409      0.711      0.724      0.677      0.209      0.886      0.902       0.93      0.925
   task    F1     P     R  MAP@50
0   box 0.717 0.711 0.724   0.677
1  pose 0.894 0.886 0.902   0.930


In [6]:
print("MERGED BIAS")
ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_merged_bias, args),
    False,
)

print("MERGED BIAS REDUCED IOU")
ultralytics_validator_wrapper.validate_model(
    args_reduced_iou,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_merged_bias_iou, args_reduced_iou),
    False,
)

MERGED BIAS


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.21s/it]


                   all        148        409        0.5      0.548      0.441      0.103      0.869      0.895      0.919      0.915
   task    F1     P     R  MAP@50
0   box 0.523 0.500 0.548   0.441
1  pose 0.882 0.869 0.895   0.919
MERGED BIAS REDUCED IOU


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]
tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.16s/it]


                   all        148        409      0.526      0.504      0.434      0.104       0.94        0.9      0.947      0.943
   task    F1     P     R  MAP@50
0   box 0.515 0.526 0.504   0.434
1  pose 0.919 0.940 0.900   0.947
